<a href="https://colab.research.google.com/github/mdshahadat786/Student-Stress-Detection-FL/blob/main/Student_Stress_FL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install flwr[simulation] scikit-learn pandas numpy streamlit plotly datasets -q
print("Setup done! ✅")


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.4/71.4 MB 18.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 99.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 109.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.7/6.7 MB 108.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 323.5/323.5 kB 45.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 102.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 119.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 251.7/251.7 kB 20.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 47.4/47.4 kB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 752.9/752.9 kB 51.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
pydrive2 1.21.3 requires cryptograph

In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

np.random.seed(42)
n_samples = 1000
data = {
    'sleep_hours': np.random.randint(4, 10, n_samples),
    'anxiety_score': np.random.randint(0, 5, n_samples),
    'academic_pressure': np.random.choice(['Low', 'Medium', 'High'], n_samples),
    'stress_label': np.random.choice(['Low', 'Medium', 'High'], n_samples, p=[0.4, 0.4, 0.2])
}
df = pd.DataFrame(data)
le_pressure = LabelEncoder()
le_label = LabelEncoder()
df['academic_pressure'] = le_pressure.fit_transform(df['academic_pressure'])
df['stress_label'] = le_label.fit_transform(df['stress_label'])
train, test = train_test_split(df, test_size=0.2, random_state=42)
train.to_csv('train.csv', index=False)
print(df.head())
print("Data ready! Shape:", df.shape)


   sleep_hours  anxiety_score  academic_pressure  stress_label
0            7              4                  1             1
1            8              3                  1             0
2            6              2                  2             1
3            8              0                  2             2
4            8              2                  2             0
Data ready! Shape: (1000, 4)


In [3]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

model = RandomForestClassifier(n_estimators=100, random_state=42)
X_train = train.drop('stress_label', axis=1)
y_train = train['stress_label']
model.fit(X_train, y_train)
X_test = test.drop('stress_label', axis=1)
y_test = test['stress_label']
preds = model.predict(X_test)
print("Centralized Accuracy:", accuracy_score(y_test, preds))
print(classification_report(y_test, preds))
import joblib
joblib.dump(model, 'stress_model.pkl')


Centralized Accuracy: 0.33
              precision    recall  f1-score   support

           0       0.12      0.05      0.07        44
           1       0.37      0.45      0.40        78
           2       0.33      0.37      0.35        78

    accuracy                           0.33       200
   macro avg       0.27      0.29      0.27       200
weighted avg       0.30      0.33      0.31       200



['stress_model.pkl']

In [4]:
import flwr as fl
from typing import Dict, List, Tuple
import joblib

class StressClient(fl.client.NumPyClient):
    def __init__(self, cid: str, X, y):
        self.cid = cid
        self.model = RandomForestClassifier(n_estimators=50, random_state=42)
        self.X_train, self.y_train = X, y
        self.model.fit(X, y)

    def get_parameters(self, config):
        return []  # RF params tricky, demo ke liye skip or serialize

    def set_parameters(self, parameters):
        pass

    def fit(self, parameters, config):
        self.model.fit(self.X_train, self.y_train)
        return [], len(self.X_train), {}

    def evaluate(self, parameters, config):
        preds = self.model.predict(self.X_train)
        acc = accuracy_score(self.y_train, preds)
        return float(acc), len(self.X_train), {"accuracy": float(acc)}

def client_fn(cid: str):
    idx = int(cid) * 100
    client_data = train.iloc[idx:idx+100]
    X_c = client_data.drop('stress_label', axis=1)
    y_c = client_data['stress_label']
    return StressClient(cid, X_c, y_c)

strategy = fl.server.strategy.FedAvg(min_fit_clients=5, min_available_clients=10)
history = fl.simulation.start_simulation(
    client_fn=client_fn, num_clients=10, config=fl.server.ServerConfig(num_rounds=3),
    strategy=strategy, client_resources={"num_cpus": 1})
print("FL History:", history)


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

	Instead, use the `flwr run` CLI command to start a local simulation in your Flower app, as shown for example below:

		$ flwr new  # Create a new Flower app from a template

		$ flwr run  # Run the Flower app in Simulation Mode

	Using `start_simulation()` is deprecated.

            This is a deprecated feature. It will be removed
            entirely in future versions of Flower.
        
/usr/local/lib/python3.12/dist-packages/jupyter_client/session.py:203: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  return datetime.utcnow().replace(tzinfo=utc)
	Instead, use the `flwr run` CLI command to start a local simulation in your Flower app, as shown for example below:

		$ flwr new  # Create a new Flower app from a template

		$ flwr run  # Run the Flower app in Simulation Mode

	Using `start_simulation()` is deprecated.

          

FL History: History (loss, distributed):
	round 1: 0.74875
	round 2: 0.74875
	round 3: 0.74875



In [6]:
%%writefile app.py
import streamlit as st
import joblib
import pandas as pd

model = joblib.load('stress_model.pkl')
le_pressure = joblib.load('le_pressure.pkl')  # Save earlier if needed
le_label = joblib.load('le_label.pkl')

st.title("🧠 Student Stress Detector (Federated)")
sleep = st.slider("Sleep Hours", 4, 10, 7)
anxiety = st.slider("Anxiety Score", 0, 4, 2)
pressure = st.selectbox("Academic Pressure", ['Low', 'Medium', 'High'])

if st.button("Predict Stress"):
    input_df = pd.DataFrame({'sleep_hours': [sleep], 'anxiety_score': [anxiety],
                             'academic_pressure': [le_pressure.transform([pressure])[0]]})
    pred = model.predict(input_df)[0]
    stress = le_label.inverse_transform([pred])[0]
    st.success(f"Your Stress: **{stress}** – { 'Rest lo!' if stress=='High' else 'Good!' }")


Overwriting app.py
